In [24]:
import pandas as pd
from datetime import date, timedelta
import numpy as np

yesterday = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")

file_path1 = f"검증데이터/({yesterday})검증.csv"
file_path2= f"예측데이터/({yesterday})예측.csv"
val_df = pd.read_csv(file_path1)
predict_df = pd.read_csv(file_path2)

In [25]:
predict_df

,종목,상승확률,상승여부
0,3M,1.80,하락
1,A- O- Smith,3.71,하락
2,Abbott Laboratories,58.01,상승
3,AbbVie,99.08,상승
4,Accenture,0.00,하락
...,...,...,...
498,Xylem Inc-,7.87,하락
499,Yum! Brands,75.38,상승
500,Zebra Technologies,7.04,하락
501,Zimmer Biomet,73.98,상승


In [26]:

predict_df1 = predict_df[["종목","상승여부"]]
# "종목"을 기준으로 병합 (inner join이면 양쪽에 모두 있는 종목만 비교)
merged_df = pd.merge(predict_df1, val_df, on="종목")

# 일치 여부 판단
merged_df["일치여부"] = merged_df["상승여부"] == merged_df["실제상승여부"]
merged_df["일치여부"] = merged_df["일치여부"].map({True: "O", False: "X"})

# 결과 예시 출력
merged_df


,종목,상승여부,실제상승여부,일치여부
0,3M,하락,상승,X
1,A- O- Smith,하락,상승,X
2,Abbott Laboratories,상승,상승,O
3,AbbVie,상승,상승,O
4,Accenture,하락,상승,X
...,...,...,...,...
498,Xylem Inc-,하락,상승,X
499,Yum! Brands,상승,하락,X
500,Zebra Technologies,하락,상승,X
501,Zimmer Biomet,상승,상승,O


In [27]:
# "O"의 개수 세기
num_matches = (merged_df["일치여부"] == "O").sum()

# 전체 비교한 행 수
total_rows = len(merged_df)

# 정확도 계산
accuracy = num_matches / total_rows

# 출력
print(f"일치 수: {num_matches} / 전체: {total_rows}")
print(f"정확도: {accuracy:.2%}")

일치 수: 222 / 전체: 503
정확도: 44.14%


In [ ]:
# 예측이 '상승'인 행만 필터링
예측_상승 = merged_df[merged_df["상승여부"] == "상승"]

# 그 중 실제도 '상승'인 행의 수

정답_상승 = 예측_상승[예측_상승["실제상승여부"] == "상승"].shape[0]
정답_하락 = 예측_상승[예측_상승["실제상승여부"] == "하락"].shape[0]
# 전체 예측이 '상승'인 개수
전체_예측_상승 = 예측_상승.shape[0]

# 정확도 계산
if 전체_예측_상승 > 0:
    상승정확도 = 정답_상승 / 전체_예측_상승
    print(f"📈 예측이 '상승'일 때 실제도 '상승'인 비율: {상승정확도:.2%}")
else:
    print("예측이 '상승'인 종목이 없습니다.")
    
if 전체_예측_상승 > 0:
    하락 = 정답_하락 / 전체_예측_상승
    print(f"📈 예측이 '상승'일 때 실제도 '하락'인 비율: {하락:.2%}")
else:
    print("예측이 '상승'인 종목이 없습니다.")

📈 예측이 '상승'일 때 실제도 '상승'인 비율: 90.87%
📈 예측이 '상승'일 때 실제도 '하락'인 비율: 9.13%


In [ ]:
path1 = f"예측데이터/({yesterday})예측.csv"
path2 = f"뉴스데이터/({yesterday})뉴스.csv"
df1 = pd.read_csv(path1)
df2 = pd.read_csv(path2)
merged = pd.merge(df1, df2, on="종목", how="inner")

merged["종합점수"] = merged["상승확률"] + merged["감정점수"]
merged["종합확률"] = np.where(merged["종합점수"] > 30, "상승", "하락")
merged

,종목,상승확률,상승여부,감정점수,종합점수,종합확률
0,3M,1.80,하락,29.228573,31.028573,상승
1,A- O- Smith,3.71,하락,-22.347259,-18.637259,하락
2,Abbott Laboratories,58.01,상승,53.979084,111.989084,상승
3,AbbVie,99.08,상승,3.189669,102.269669,상승
4,Accenture,0.00,하락,23.580737,23.580737,하락
...,...,...,...,...,...,...
498,Xylem Inc-,7.87,하락,0.000000,7.870000,하락
499,Yum! Brands,75.38,상승,0.000000,75.380000,상승
500,Zebra Technologies,7.04,하락,0.000000,7.040000,하락
501,Zimmer Biomet,73.98,상승,0.000000,73.980000,상승


In [37]:

merged1 = merged[["종목","종합확률"]]

merged0 = pd.merge(merged1, val_df, on="종목")

# 일치 여부 판단
merged0["일치여부"] = merged0["종합확률"] == merged0["실제상승여부"]
merged0["일치여부"] = merged0["일치여부"].map({True: "O", False: "X"})

# 결과 예시 출력
merged_df

,종목,상승여부,실제상승여부,일치여부
0,3M,하락,상승,X
1,A- O- Smith,하락,상승,X
2,Abbott Laboratories,상승,상승,O
3,AbbVie,상승,상승,O
4,Accenture,하락,상승,X
...,...,...,...,...
498,Xylem Inc-,하락,상승,X
499,Yum! Brands,상승,하락,X
500,Zebra Technologies,하락,상승,X
501,Zimmer Biomet,상승,상승,O


In [38]:
# "O"의 개수 세기
num_matches = (merged0["일치여부"] == "O").sum()

# 전체 비교한 행 수
total_rows = len(merged0)

# 정확도 계산
accuracy = num_matches / total_rows

# 출력
print(f"일치 수: {num_matches} / 전체: {total_rows}")
print(f"정확도: {accuracy:.2%}")

일치 수: 260 / 전체: 503
정확도: 51.69%


In [39]:
# 예측이 '상승'인 행만 필터링
예측_상승 = merged0[merged0["종합확률"] == "상승"]

# 그 중 실제도 '상승'인 행의 수

정답_상승 = 예측_상승[예측_상승["실제상승여부"] == "상승"].shape[0]
정답_하락 = 예측_상승[예측_상승["실제상승여부"] == "하락"].shape[0]
# 전체 예측이 '상승'인 개수
전체_예측_상승 = 예측_상승.shape[0]

# 정확도 계산
if 전체_예측_상승 > 0:
    상승정확도 = 정답_상승 / 전체_예측_상승
    print(f"📈 예측이 '상승'일 때 실제도 '상승'인 비율: {상승정확도:.2%}")
else:
    print("예측이 '상승'인 종목이 없습니다.")
    
if 전체_예측_상승 > 0:
    하락 = 정답_하락 / 전체_예측_상승
    print(f"📈 예측이 '상승'일 때 실제도 '하락'인 비율: {하락:.2%}")
else:
    print("예측이 '상승'인 종목이 없습니다.")

📈 예측이 '상승'일 때 실제도 '상승'인 비율: 91.24%
📈 예측이 '상승'일 때 실제도 '하락'인 비율: 8.76%
